# Train the unproofread axon/dendrite classifier

The goal of this model is to recover the dendrite of the cell at the soma from an object that may hold merges. It is the companion of `train_compartment_model.ipynb`, which trains the model for proofread cells.

- **Features are local.** Downstream windows of 10 and 30 µm (30 µm is about the reach of 15 hops), counts normalized by area and length, and nothing measured against the whole object (`LOCAL_FEATURE_SPEC`, `LOCAL_WINDOWS`).
- **Training cells are proofread cells with merges added.** Each cell is used once as it is, and once with pieces of other proofread cells' axons attached to its dendrites at random places. Labels on the cell come from synapse flow; an attached piece is axon.
- **The score is the dendrite.** After the tree decode: how much of the cell's own dendrite is called dendrite, and how much of what is called dendrite is something else.

In [1]:
import os
from pathlib import Path

DATASET = "minnie65"   # or "v1dd"

# Where s1_skeletons wrote the cells, and the materialization whose root ids they are named by.
SOURCES = {
    "minnie65": dict(datastack="minnie65_public", version=1822, cells="gs://csm-skel/osy/minnie65/skel", bundled_model="minnie65_ds15_us0_bd0.json"),
    "v1dd": dict(datastack="v1dd", version=1643, cells="gs://csm-skel/osy/v1dd/skel", bundled_model="v1dd_ds15_us0_bd0.json"),
}
SOURCE = SOURCES[DATASET]

# Feature tables and the trained model are written here; cells are read from the bucket each time.
CACHE_DIR = Path.home() / ".cache" / "ossify_compartment_training" / f"{DATASET}_v{SOURCE['version']}_unproofread"

MAX_CELLS = int(os.environ.get("MAX_CELLS", 0)) or None  # None = every proofread cell in the bucket
N_WORKERS = 16                # cells handled at once
SEED = 0

# A cell contributes only if synapse flow split it cleanly.
QC = dict(max_pre_off_fraction=0.1, max_post_off_fraction=0.1, min_synapses=100)

# Merges added to each cell.
N_GRAFTS = 10                     # axon pieces attached to each cell's dendrite
N_FRAGMENTS_PER_DONOR = 5         # pieces cut from each cell's axon for use on other cells
FRAGMENT_CABLE_UM = (20, 300)     # cable in a piece

ROWS_PER_CELL = 3000          # vertices sampled from each cell as it is
HOLDOUT_FRACTION = 0.2        # of cells

# The tree decode, as s1_skeletons runs it. Saved with the model.
COSTS = {("axon", "dendrite"): 50, ("dendrite", "axon"): 2}
ABSORB_MIN_SIZE = 10

In [2]:
import json
import logging
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import xgboost
from caveclient import CAVEclient
from cloudfiles import CloudFiles

import ossify
from ossify import algorithms
from ossify.base import Link
from ossify.compartments import (
    LOCAL_FEATURE_SPEC,
    LOCAL_WINDOWS,
    ProbaVertexModel,
    StructuredLabeler,
    make_skel_prop_df,
)
from ossify.structured_prediction import TransitionSchema
from ossify.translate import SWCCompartment

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

FEATURE_DIR = CACHE_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

cell_files = ossify.CellFiles(SOURCE["cells"])

def load(root_id):
    return cell_files.load(f"{root_id}.osy")

client = CAVEclient(SOURCE["datastack"])
print(DATASET, SOURCE["datastack"], "v", SOURCE["version"], "cells from", SOURCE["cells"])

minnie65 minnie65_public v 1822 cells from gs://csm-skel/osy/minnie65/skel


## 1. Cells

In [3]:
# Queried at the version's timestamp, which still works after the version itself has expired.
proof = client.materialize.live_query(
    "proofreading_status_and_strategy", timestamp=client.materialize.get_timestamp(SOURCE["version"])
)
proof = proof[proof.status_axon & proof.status_dendrite].drop_duplicates("pt_root_id")
proofread = proof.pt_root_id.astype("int64").to_numpy()

in_bucket = CloudFiles(SOURCE["cells"]).exists([f"{r}.osy" for r in proofread])
root_ids = np.array([r for r in proofread if in_bucket[f"{r}.osy"]])
print(len(proofread), "cells with axon and dendrite proofread;", len(root_ids), "of them in the bucket")
if MAX_CELLS:
    root_ids = np.random.default_rng(SEED).choice(root_ids, MAX_CELLS, replace=False)

2220 cells with axon and dendrite proofread; 2219 of them in the bucket


## 2. Grafting

`cut_fragments` takes random pieces of a cell's axon. Each piece is a whole subtree, with its L2 vertices and synapses. `graft` attaches pieces to random dendrite vertices of another cell and returns an ordinary `Cell`, so features and the decode run on it exactly as they would on a real object. With no pieces, `graft` rebuilds the cell unchanged.

In [4]:
GRAPH_COLUMNS = ["area_nm2", "size_nm3", "max_dt_nm"]
AXON = int(SWCCompartment.AXON)
DENDRITE = [int(SWCCompartment.DENDRITE), int(SWCCompartment.APICAL_DENDRITE)]

def parts(cell):
    """The plain tables a cell is rebuilt from."""
    skel, graph = cell.skeleton, cell.graph
    return dict(
        xyz=np.asarray(skel.vertices, dtype=float),
        edges=np.asarray(skel.edges_positional),
        root=int(skel.root_positional),
        compartment=np.asarray(skel.get_feature("compartment")),
        graph=graph.nodes[list(graph.spatial_columns) + GRAPH_COLUMNS],
        to_skel=np.asarray(graph.map_index_to_layer("skeleton", as_positional=True)),
        # L2 ids as int64 like the graph's index: matching uint64 against int64 goes through
        # floats and pairs up ids that differ only in their last digits.
        pre=cell.annotations["pre_syn"].nodes.astype({"pre_pt_l2_id": "int64"}),
        post=cell.annotations["post_syn"].nodes.astype({"post_pt_l2_id": "int64"}),
    )

def cut_fragments(cell, rng, n=N_FRAGMENTS_PER_DONOR, cable_um=FRAGMENT_CABLE_UM):
    """Random pieces of a cell's axon, each a whole subtree with its L2 vertices and synapses."""
    skel, p = cell.skeleton, parts(cell)
    below = algorithms.subtree_aggregate(skel, np.asarray(skel.half_edge_length, dtype=float), agg="sum") / 1000
    tops = np.flatnonzero((p["compartment"] == AXON) & (below >= cable_um[0]) & (below <= cable_um[1]))
    out, taken = [], np.zeros(len(p["xyz"]), dtype=bool)
    for top in rng.permutation(tops):
        if len(out) == n:
            break
        keep = np.asarray(skel.downstream_vertices(int(top), inclusive=True, as_positional=True))
        if taken[keep].any():
            continue  # pieces must not share vertices, or two could not go on one cell
        taken[keep] = True
        new = np.full(len(p["xyz"]), -1)
        new[keep] = np.arange(len(keep))
        in_graph = new[p["to_skel"]] >= 0
        l2_ids = p["graph"].index[in_graph]
        out.append(dict(
            xyz=p["xyz"][keep],
            edges=new[p["edges"][(new[p["edges"]] >= 0).all(axis=1)]],
            graph=p["graph"][in_graph],
            to_skel=new[p["to_skel"][in_graph]],
            pre=p["pre"][p["pre"].pre_pt_l2_id.isin(l2_ids)],
            post=p["post"][p["post"].post_pt_l2_id.isin(l2_ids)],
        ))
    return out

def graft(cell, fragments, rng):
    """A copy of the cell with each fragment attached to a random dendrite vertex.

    A fragment is attached at a random one of its own vertices, as a passing axon touches
    a dendrite, and moved so that vertex sits on the dendrite. The copy's skeleton carries
    `compartment` (the fragments are axon) and `grafted`.
    """
    host = parts(cell)
    dendrite = np.flatnonzero(np.isin(host["compartment"], DENDRITE))
    xyz, edges, to_skel = [host["xyz"]], [host["edges"]], [host["to_skel"]]
    graph, pre, post = [host["graph"]], [host["pre"]], [host["post"]]
    compartment, grafted = [host["compartment"]], [np.zeros(len(host["xyz"]), dtype=bool)]
    offset = len(host["xyz"])
    for f in fragments:
        onto, contact = int(rng.choice(dendrite)), int(rng.integers(len(f["xyz"])))
        shift = host["xyz"][onto] - f["xyz"][contact]
        xyz.append(f["xyz"] + shift)
        edges += [f["edges"] + offset, np.array([[contact + offset, onto]])]
        to_skel.append(f["to_skel"] + offset)
        graph.append(f["graph"])
        pre.append(f["pre"]); post.append(f["post"])
        compartment.append(np.full(len(f["xyz"]), AXON)); grafted.append(np.ones(len(f["xyz"]), dtype=bool))
        offset += len(f["xyz"])
    graph = pd.concat(graph)
    merged = (
        ossify.Cell(name=cell.name)
        .add_graph(vertices=graph.reset_index(), spatial_columns=list(cell.graph.spatial_columns), edges=[], vertex_index="l2_id", edges_as_positional=False)
        .add_skeleton(
            vertices=np.concatenate(xyz), edges=np.concatenate(edges), root=host["root"],
            features={"compartment": np.concatenate(compartment), "grafted": np.concatenate(grafted)},
            linkage=Link(mapping=np.concatenate(to_skel), source="graph", map_value_is_index=False),
        )
    )
    for side, tables in (("pre", pre), ("post", post)):
        merged.add_point_annotations(
            f"{side}_syn", vertices=pd.concat(tables), spatial_columns=cell.annotations[f"{side}_syn"].spatial_columns,
            linkage=Link(mapping=f"{side}_pt_l2_id", target="graph"),
        )
    return merged

## 3. Labels, QC and fragments

One pass over the cells measures how cleanly synapse flow split each one and cuts its axon pieces. Cells are then divided into training and held-out sets, and a cell only ever receives pieces from its own side, so no held-out axon is seen in training.

In [5]:
def is_axon_of(cell):
    return np.asarray(cell.skeleton.get_feature("compartment")) == AXON

def qc_and_fragments(root_id):
    cell = load(root_id)
    is_axon = is_axon_of(cell)
    row = algorithms.synapse_compartment_qc(cell, is_axon, **QC)
    compartment = cell.skeleton.get_feature("compartment")
    row.update(
        root_id=int(root_id),
        has_soma=bool(compartment[cell.skeleton.root_positional] == int(SWCCompartment.SOMA)),
        axon_fraction=float(is_axon.mean()),
    )
    row["use"] = bool(row["passes"] and row["has_soma"] and 0.02 <= row["axon_fraction"] <= 0.98)
    rng = np.random.default_rng([SEED, int(root_id) % 2**32, 0])
    return row, cut_fragments(cell, rng) if row["use"] else []

with ThreadPoolExecutor(N_WORKERS) as pool:
    results = list(pool.map(qc_and_fragments, root_ids))
qc = pd.DataFrame([row for row, _ in results]).set_index("root_id")

use_ids = np.random.default_rng(SEED).permutation(qc.index[qc.use].to_numpy())
n_test = int(round(len(use_ids) * HOLDOUT_FRACTION))
test_ids, train_ids = use_ids[:n_test], use_ids[n_test:]
side_of = {**{r: "test" for r in test_ids}, **{r: "train" for r in train_ids}}

# Pieces available to each side, as (donor, piece).
bank = {"train": [], "test": []}
for root_id, (_, fragments) in zip(root_ids, results):
    if root_id in side_of:
        bank[side_of[root_id]] += [(int(root_id), f) for f in fragments]
del results

print(int(qc.use.sum()), "of", len(qc), "cells pass;", len(train_ids), "train,", len(test_ids), "held out")
print("axon pieces:", {k: len(v) for k, v in bank.items()}, " median vertices per piece:", int(np.median([len(f["xyz"]) for _, f in bank["train"]])))

1969 of 2219 cells pass; 1575 train, 394 held out
axon pieces: {'train': 7873, 'test': 1968}  median vertices per piece: 54


## 4. Features

Each cell gives two sets of rows. `clean` rows are a sample of its vertices as it is. The grafted copy adds only the rows that are new: the attached pieces (`fragment`) and the cell's own vertices whose windows now reach a piece (`near graft`). The rest of the grafted copy is identical to the clean cell and is not repeated.

The cached rows depend on which cells are in each set and on the graft settings. Delete `FEATURE_DIR` after changing the cells, the QC or the graft parameters.

In [6]:
def features_of(cell):
    return make_skel_prop_df(
        cell, downstream_hops=0, feature_spec=LOCAL_FEATURE_SPEC, windows=LOCAL_WINDOWS
    ).reset_index(drop=True)

def grafted(cell, root_id):
    """The cell with N_GRAFTS axon pieces from other cells on its side. The same pieces every time."""
    rng = np.random.default_rng([SEED, int(root_id) % 2**32, 1])
    others = [f for donor, f in bank[side_of[root_id]] if donor != root_id]
    chosen = rng.choice(len(others), min(N_GRAFTS, len(others)), replace=False)
    return graft(cell, [others[i] for i in chosen], rng)

def cell_features(root_id):
    """Clean and grafted rows for one cell, cached as parquet."""
    path = FEATURE_DIR / f"{root_id}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    cell = load(root_id)
    n, root = cell.skeleton.n_vertices, cell.skeleton.root_positional

    clean = features_of(cell)
    columns = list(clean.columns)
    merged_cell = grafted(cell, root_id)
    merged = features_of(merged_cell)
    is_fragment = np.asarray(merged_cell.skeleton.get_feature("grafted"))
    sees_graft = ~np.isclose(
        clean.to_numpy(dtype=float), merged.iloc[:n].to_numpy(dtype=float), equal_nan=True
    ).all(axis=1)

    clean = clean.assign(is_axon=is_axon_of(cell), kind="clean").drop(index=root)  # the soma is neither
    clean = clean.sample(min(len(clean), ROWS_PER_CELL), random_state=SEED)
    merged = merged.assign(
        is_axon=is_axon_of(merged_cell), kind=np.where(is_fragment, "fragment", "near graft")
    )
    merged = merged[is_fragment | np.r_[sees_graft, np.zeros(len(merged) - n, dtype=bool)]].drop(index=root, errors="ignore")
    out = pd.concat([clean, merged], ignore_index=True)[columns + ["is_axon", "kind"]]
    out["root_id"] = int(root_id)
    out.to_parquet(path)
    return out

# One cell at a time: the feature code holds Python's lock, so threads here only slow it down
# (0.7 s per cell alone, 2.4 s per cell with 16 threads).
table = pd.concat(map(cell_features, use_ids), ignore_index=True)
META = ["is_axon", "kind", "root_id"]
FEATURES = [c for c in table.columns if c not in META]
is_test = table.root_id.isin(test_ids)
train, test = table[~is_test], table[is_test].copy()
print(len(table), "rows from", table.root_id.nunique(), "cells;", len(FEATURES), "features")
table.groupby("kind").is_axon.agg(rows="size", axon="mean").style.format({"axon": "{:.0%}"})

7291046 rows from 1969 cells; 33 features


,rows,axon
kind,,
clean,5798947,68%
fragment,1209760,100%
near graft,282339,0%


## 5. Train

Two models on the same cells: one sees only the clean rows, as before, and one also sees the grafted rows. The comparison says whether adding merges to training is worth it.

In [7]:
def fit(rows):
    model = xgboost.XGBClassifier(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.5,
        min_child_weight=5,
        scale_pos_weight=float((~rows.is_axon).sum() / max(rows.is_axon.sum(), 1)),
        early_stopping_rounds=50,
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
    )
    held_out = test if "fragment" in set(rows.kind) else test[test.kind == "clean"]
    model.fit(rows[FEATURES], rows.is_axon, eval_set=[(held_out[FEATURES], held_out.is_axon)], verbose=False)
    return model

models = {"clean only": fit(train[train.kind == "clean"]), "with grafts": fit(train)}
print({name: m.best_iteration + 1 for name, m in models.items()}, "trees")

{'clean only': 827, 'with grafts': 1108} trees


## 6. Evaluate

In [8]:
gain = pd.Series(models["with grafts"].get_booster().get_score(importance_type="total_gain"))
(gain / gain.sum()).sort_values(ascending=False).head(15).rename("share of total gain").to_frame().style.format("{:.1%}")

,share of total gain
down30um_syn_in_per_um,39.1%
down30um_syn_in_per_area,26.3%
down10um_syn_in_per_um,10.8%
down10um_syn_in_per_area,7.7%
down30um_syn_in,6.7%
down10um_vol_um3,3.9%
syn_in_per_area,0.8%
down30um_area_um2,0.7%
down10um_area_um2,0.7%
down30um_cable_um,0.5%


### Per vertex, before the decode

Share of held-out vertices the classifier gets wrong, by what the vertex is. `near graft` rows are the cell's own vertices within reach of an attached piece.

In [9]:
test["truth"] = np.where(test.is_axon, "axon", "dendrite")
wrong = pd.DataFrame({
    name: (m.predict(test[FEATURES]).astype(bool) != test.is_axon).groupby([test.kind, test.truth]).mean()
    for name, m in models.items()
})
wrong.insert(0, "vertices", test.groupby(["kind", "truth"]).size())
wrong.style.format({name: "{:.2%}" for name in models})

### The dendrite, after the decode

Whole held-out cells, as they are and with grafts, labelled by each model and then decoded (Viterbi on the skeleton with the costs above, then absorbing small compartments). `bundled` is the 15-hop model the pipeline uses now.

- **dendrite recovered**: share of the cell's own dendrite cable that is called dendrite.
- **not dendrite**: share of the cable called dendrite that is really the cell's axon or an attached piece.
- **pieces missed**: share of attached cable that is called dendrite.

In [10]:
schema = TransitionSchema(classes=["dendrite", "axon"], transitions=COSTS, root_classes=["dendrite"])
labelers = {"bundled": StructuredLabeler(ProbaVertexModel.from_config(SOURCE["bundled_model"]), schema, absorb_min_size=ABSORB_MIN_SIZE)}
for name, m in models.items():
    encoder = ProbaVertexModel(m, FEATURES, feature_set="local")
    labelers[name] = StructuredLabeler(encoder, schema, absorb_min_size=ABSORB_MIN_SIZE)

def dendrite_scores(root_id):
    cell = load(root_id)
    rows = []
    for condition, c in (("clean", cell), ("grafted", grafted(cell, root_id))):
        skel = c.skeleton
        cable = np.asarray(skel.half_edge_length, dtype=float)
        cable[skel.root_positional] = 0
        own_dendrite = ~is_axon_of(c)
        piece = np.asarray(skel.get_feature("grafted")) if condition == "grafted" else np.zeros(skel.n_vertices, dtype=bool)
        for name, labeler in labelers.items():
            called = np.asarray(labeler.predict(c)) == "dendrite"
            rows.append({
                "root_id": int(root_id), "cell": condition, "model": name,
                "dendrite recovered": cable[called & own_dendrite].sum() / cable[own_dendrite].sum(),
                "not dendrite": cable[called & ~own_dendrite].sum() / max(cable[called].sum(), 1),
                "pieces missed": cable[called & piece].sum() / cable[piece].sum() if piece.any() else np.nan,
            })
    return rows

scores = pd.DataFrame([row for root_id in test_ids[:50] for row in dendrite_scores(root_id)])
scores.groupby(["cell", "model"])[["dendrite recovered", "not dendrite", "pieces missed"]].agg(["mean", "min", "max"]).style.format("{:.2%}")

## 7. Save

Saves the model trained with grafts. Written into `src/ossify/compartment_models`, in the same layout as the bundled models, so it loads with `load_labeler("<dataset>_local")`.

In [11]:
from ossify.compartments import MODEL_DIR

# Saved beside the bundled models, in the same layout, so it loads by name.
name = f"{DATASET}_local"
models["with grafts"].save_model(MODEL_DIR / "xgb" / f"{name}.ubj")
(MODEL_DIR / f"{name}.json").write_text(json.dumps({
    "dataset": DATASET,
    "description": "Axon/dendrite on objects that may hold merges. Downstream windows of 10 and 30 um only; trained with axon pieces grafted onto dendrites.",
    "downstream_hops": 0,
    "upstream_hops": 0,
    "bidirectional_hops": 0,
    "feature_set": "local",
    "feature_columns": FEATURES,
    "model_file": f"xgb/{name}.ubj",
    "decode": {"schema": schema.to_dict(), "absorb_min_size": ABSORB_MIN_SIZE},
    "training": {
        "datastack": SOURCE["datastack"],
        "version": SOURCE["version"],
        "cells": SOURCE["cells"],
        "qc": QC,
        "n_train_cells": int(len(train_ids)),
        "grafts": dict(per_cell=N_GRAFTS, cable_um=FRAGMENT_CABLE_UM),
    },
}, indent=2) + "\n")
print("saved; load with load_labeler(" + repr(name) + ")")

saved; load with ProbaVertexModel.from_config('minnie65_local')
